# AI 활용에 관하여

과제는 AI 를 사용할 것을 전제로 만들어지고 있습니다.
하지만, 학습의 효율성을 위해서 무분별한 AI 사용은 권장드리지 않고 있어요.

따라서, AI 를 피어리뷰어로서 생각하고 사용해주시는 것을 권고드립니다.  
이를 판단할 수 있는 하나의 기준으로는 같은 수업을 듣는 다른 친구에게 물어볼 수 있는 질문인가? 라고 스스로 물어보는 것도 좋은 기준이 될 수 있을 것 같습니다.

아래는 이에 대한 몇가지 예시입니다.


> AI 사용 가능:
> - 문법 및 API 확인
> - 오류 메시지 해석
> - 실행 환경 문제 해결
> - 내가 작성한 설명의 반례 요청

> AI에 맡기지 않을 것:
> - 실행 전 예측
> - 실행 결과에서 근거 선택
> - 의견 및 선택의 최종 판단

# 3주차 과제: 이미지의 공간 구조와 CNN

1~2주차를 거치면서, 딥러닝이 어떻게 추론하고 학습하는지를 알아보았습니다.   
이번에는 구체적으로 이미지라는 영역에서 CNN 이라는 아키텍처가 어떻게 처리하는지 알아봅시다.  

**필수 범위** 
* 기본 개념과 이미지의 구조(Q1~Q7)
* 2차원 연산과 채점머신(Q8~Q10)
* LeNet, AlexNet을 참고한 Fashion-MNIST 모델(Q11~Q13)  


**선택 심화**
* 여러 채널과 kernel bank 살펴보기
* NumPy 계산과 Keras Custom Layer의 가중치 업데이트 살펴보기

## 목차

1. [스터디 내용 복습하기](#section-1-review)
2. [이미지의 구조](#section-2-image-domain)
3. [작은 이미지로 연산 직접 구현하기](#section-3-operations)
4. [LeNet, AlexNet에서 Fashion-MNIST로](#section-4-models)
5. [선택 심화: 여러 채널과 kernel bank](#advanced-channels)
6. [선택 심화: Custom Layer와 가중치 업데이트](#advanced)

## 실행 환경

1주차, 2주차와 같은 NumPy, TensorFlow/Keras 환경을 사용합니다. 아래 셀은 연산 실습과 모델 학습에 필요한 라이브러리를 불러옵니다.  
Fashion-MNIST는 Keras에서 불러오며, 처음 실행하는 환경에서는 데이터 다운로드가 필요할 수 있습니다.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)

<a id="section-1-review"></a>
# 1. 스터디 내용 복습하기

스터디에서 배운 내용을 정리해봅시다.

**Q1.** 1주차 Fashion-MNIST 모델의 `Flatten`은 28x28 이미지를 어떤 형태로 바꾸었나요?  
그 뒤 `Dense`의 뉴런 하나는 입력의 어느 부분과 연결되나요?

**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

**Q2.** Dense Layer와 CNN을 비교해봅시다.  
CNN의 작은 필터를 통한 계산은 `Dense`의 뉴런 계산과 어떻게 다른가요?  
Dense는 입력의 어느 부분까지 연결될 수 있나요?  

HINT: Dense 에서는 Flatten 을 통해 Shape 를 조절해서 사용했습니다.  
이로 인해 잃어버리는 것이 무엇일까요? 이에 관해서도 같이 정리해보세요.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

**Q3.** 28x28 흑백 이미지 한 장에 3x3 kernel **한 개**를 적용한다고 가정해봅시다.  
stride는 1, zero padding은 한 줄(`padding=1`)이며 bias는 사용하지 않습니다. 이 kernel은 출력의 몇 위치에서 사용되나요?  

학습되는 parameter는 어떤 값이며 몇 개인가요? 이 예시에서 미리 정한 `kernel_size`, `strides`, `padding` 설정은 왜 hyperparameter인가요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

**Q4.** `padding`, `stride`, `max pooling` 에 대해서, 각 연산을 사용하면 어떻게 되나요? 각 연산을 왜 사용하나요?  
예시가 필요하다면, 28x28 이미지에 대해 적용한다고 가정하고 이야기해주세요.

**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

**Q5.** 아래 조건에서 예상되는 출력의 크기를 정리해주세요.

| 입력 | kernel | zero padding | stride | 예상 출력 크기 |
|---|---:|---:|---:|---|
| 28x28 | 3x3 | 0 | 1 | |
| 28x28 | 3x3 | 1 | 1 | |
| 28x28 | 3x3 | 1 | 2 | |

첫 행의 feature map에 2x2 max pooling, stride 2, padding 없음(`valid`)을 적용하면 크기가 어떻게 되나요?  
단순 공식으로 해결해볼수도 있지만, 머리속에서 어떤식으로 연산이 진행되는지 생각해보며 해결해주세요.

**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

<a id="section-2-image-domain"></a>
# 2. 이미지의 구조

모든 문제를 Dense Layer 만으로도 해결할 수 있을 것입니다.  
Dense Layer 는 어떤 문제에 대해서든 적용시킬 수 있습니다.  
이미지에 대해서도, 동영상에 대해서도, 언어에 대해서도, 숫자로 변환시킬 수 있다면 모두 적용할 수 있습니다.  

이를 "모델의 표현력이 좋다" 라고 표현하곤 합니다.  
(1주차 과제의 심화문제에서 퍼셉트론만으로 모든 함수를 근사할 수 있다고 이야기한 것과 유사한 내용입니다.)

하지만 각 영역, 도메인에 따라 다른 구조를 사용하게 됩니다.  
그 이유는 Dense Layer 가 모든 도메인을 표현할 수 있는 것은 맞지만, 정확한 표현을 위해 필요한 데이터가 너무 많기 때문입니다.  

즉, 표현력을 조금 희생하지만 모델의 정확도를 빠르게 높이기 위해 혹은, 더 적은 데이터로 효율적으로 학습하기 위해,  
저희는 일종의 가정과 함께(Inductive Bias) 모델의 구조를 변화시키곤 합니다.  

따라서, 이미지를 CNN 으로 처리하는 것을 알기 위해서는 이미지가 어떤 특징을 가지고 있고, 이를 CNN 이 어떻게 활용하는지를 알아봐야합니다.

### 이미지의 의미는 주변에서 온다.

스터디에서 확대된 음식 이미지를 바탕으로 음식을 맞추는 게임을 진행했었습니다.  
여러분은 어떻게 음식을 추론했나요?  
아마 부분적인 이미지에서 "숙주나물" 이라는 키워드를 생각해내고, 숙주나물이 있는 국물이 있는 음식을 통해, "라멘" 이라고 추론했다가 틀렸을겁니다.  
(저만 그랬나요?)  

뭐 비록 틀린 추론이었지만, 논리 자체는 그럴듯합니다.  
> 부분적 이미지를 바탕으로 어떤 개념을 인식하고, 그 개념을 모아서 전체 이미지를 추론하게 됩니다.  

CNN 에서도 부분적인 영역에서 어떤 특징을 가져오고(Feature Map), 이들을 모아서 추론하게 됩니다.  
Dense Layer 가 모인 Feature Map 에서 최종 판단을 내리게 됩니다.

### 이미지의 이동과 회전

이번에는 마라탕 이미지를 90° 회전했다고 해봅시다. 갑자기 라멘이 되기보다는 쏟은 마라탕으로 보일 것입니다.  
**이미지 분류**에서는 이렇게 입력을 회전하거나 조금 이동해도 정답이 같은 경우가 많습니다. 다만 방향이 정답을 결정하거나 대상이 화면 밖으로 나가면 정답이 달라질 수 있습니다.

$x$를 이미지, $R$을 회전, $f$를 분류 모델이라고 할 때, 정답이 회전 후에도 같은 경우에 바라는 관계는 다음과 같습니다.

$$
f(Rx)=f(x)
$$

**Convolution의 feature map**에는 다른 성질이 있습니다. 동일한 kernel을 모든 위치에 적용하므로, 이미지 안의 패턴을 한 칸 옮기면 그 패턴에 대한 반응 위치도 대체로 한 칸 옮겨갑니다. 이를 *이동 등변성*이라고 합니다.  
반면 세로 경계를 90° 돌리면 가로 경계가 됩니다. 같은 세로 경계 kernel의 반응이 단순히 회전한 결과가 되지는 않으므로, 일반적인 convolution만으로 회전 등변성이나 불변성이 자동으로 보장되지는 않습니다.

[첨부된 실습 HTML](./convolution_visualizer_real.html)에서 입력을 이동하거나 회전하며 반응을 비교해볼 수 있습니다.


### Convolution

이번 과제의 흑백 이미지 $x$와 3x3 kernel $K$를 생각해봅시다.  
출력의 $(i,j)$ 위치에서는 입력의 $(i,j)$를 왼쪽 위로 하는 3x3 patch와 kernel의 같은 자리끼리 곱해 아홉 값을 더합니다.

$$
y[i,j]=\sum_{u=0}^{2}\sum_{v=0}^{2} K[u,v]\,x[i+u,j+v]
$$

$x[i+u,j+v]$는 현재 보는 patch의 픽셀, $K[u,v]$는 그 자리에 곱할 가중치입니다.  
주의할 점은 $K$, 즉 Kernel 은 $i, j$ 와 무관하게 동일한 Kernel 을 사용합니다.  
즉, 모든 $(i,j)$ 에 대해서 동일한 Kernel 을 사용하게 됩니다.  

엄밀한 신호 처리 용어로는 kernel을 뒤집지 않는 *cross-correlation* 이지만, Keras를 포함한 딥러닝 문맥에서는 보통 convolution이라고 부릅니다.  
그러나 이 2개의 연산은 서로 동등하다는 점이 알려져있어 그렇구나 하고 넘어가도 충분할 듯합니다.

하나의 $y[i, j]$ 연산이 실습에서 필터가 정지한 상태에서 나온 1개의 값입니다.  
이를 모든 위치에서 실행하고 모은 결과가 Convolution 의 결과가 됩니다.

여기까지 관련 개념 복습이었습니다. 더 자세한 내용은 아래 내용을 참고해주세요.  
CNN 기초 관련 강의(과제하는 중이라면 8분정도까지만 보셔도 됩니다): https://youtu.be/bxVkZ4M-hIE?si=Eu3vMEeOHM7hR2bT  
Convolution Layer 관련 글: https://visionbook.mit.edu/convolutional_neural_nets.html

**Q6.** 이미지와 Convolution에 대해 학습한 내용을 한번 정리해봅시다.  
다음의 내용을 포함해서 작성해주세요.

* 이미지라는 도메인이 가지는 특징은 무엇인가요?
* CNN은 그 특징을 어떻게 이용하나요?  
* Dense Layer도 이미지를 학습할 수 있는데, CNN처럼 국소성과 가중치 공유를 구조에 미리 반영하지는 않는 이유는 무엇인가요?  
* 이미지를 이동하거나 회전했을 때 convolution의 feature map은 어떻게 달라지나요? 두 변화에 같은 성질이 자동으로 성립하나요?


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

<a id="section-3-operations"></a>
# 3. 작은 이미지로 연산 직접 구현하기

이미지를 처리하는 연산을 직접 구현해봅시다.  
특별한 언급이 없다면, 흑백 이미지 한 장을 2차원 배열로 입력받는다고 가정합니다.  
픽셀 값은 정수 또는 실수일 수 있습니다. Q7의 `small_image`는 0과 1을 담은 `float32` 배열입니다.

- 입력과 필터는 **2차원 NumPy 배열 한 장**입니다. 색상 channel, batch 등은 고려하지 않습니다.
- `conv3x3_valid`: **3x3 kernel, stride 1, padding 없음**, bias와 활성화함수는 고려하지 않습니다.
- `zero_pad_2d`: 이미지 둘레에 **0을 한 줄** 추가합니다.
- `max_pool2x2`: **짝수 높이, 너비**, 2x2 창, stride 2, padding 없음.
- 이번 과제의 `convolution`은 Keras의 `Conv2D`와 같이 kernel을 뒤집지 않고 입력 patch와 **같은 위치끼리 곱해 더합니다**.

**Q7.** 아래 코드를 실행해 입력 이미지와 kernel의 heatmap을 확인해주세요.  
3x3 kernel을 적용할 때 출력 크기와 `small_image[1][2]`, `small_image[0][0]`을 기준으로 한 연산 과정을 정리해주세요.  
청록색 테두리는 `[1][2]`, 주황색 테두리는 `[0][0]`을 가리킵니다. 각 좌표에 kernel의 어느 자리를 맞출지, stride와 padding을 어떻게 둘지 먼저 적어주세요. pooling을 적용한다면 그 조건도 명시해주세요.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

small_image = np.array([
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
], dtype=np.float32)

kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=np.float32)


# 그래프를 작성하는 코드입니다. 자세하게 보지 않아도 됩니다!
def show_values(ax, values, title, *, cmap, vmin, vmax):
    ax.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(f"{title} ({values.shape[0]}x{values.shape[1]})")
    ax.set_xlabel("column")
    ax.set_ylabel("row")
    ax.set_xticks(range(values.shape[1]))
    ax.set_yticks(range(values.shape[0]))
    ax.set_xticks(np.arange(-0.5, values.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, values.shape[0], 1), minor=True)
    ax.grid(which="minor", color="lightgray", linewidth=0.8)
    ax.tick_params(which="both", length=0)
    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            value = values[row, col]
            color = "white" if (value < 0.5 if cmap == "gray" else abs(value) > 0.5) else "black"
            ax.text(col, row, f"{value:g}", ha="center", va="center", color=color)


fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), layout="constrained")
show_values(axes[0], small_image, "small_image", cmap="gray", vmin=0, vmax=1)
show_values(axes[1], kernel, "kernel", cmap="coolwarm", vmin=-1, vmax=1)
axes[0].add_patch(Rectangle((2 - 0.5, 1 - 0.5), 1, 1, fill=False, edgecolor="#00A6A6", linewidth=3))
axes[0].add_patch(Rectangle((0 - 0.5, 0 - 0.5), 1, 1, fill=False, edgecolor="#E69F00", linewidth=3))
plt.show()


## Q8. Zero padding 한 줄 구현하기

`zero_pad_2d(image)`를 작성해주세요. 예를 들어 2x2 입력은 가운데에 원본 값이 들어간 4x4 배열이 됩니다.  
새로 생긴 테두리의 값은 모두 0입니다. `np.zeros`와 배열 슬라이싱을 사용해도 됩니다.

- **입력:** `(높이, 너비)` 형태의 2차원 NumPy 배열 `image`.
- **출력:** 원본 값을 가운데에 유지하고 바깥 한 줄이 모두 0인 `(높이+2, 너비+2)` 배열.

예를 들어 다음 입력은 4x4 배열로 바뀌어야 합니다.

```python
    image = np.array([[1, 2], [3, 4]])
    zero_pad_2d(image)
    # = array([[0, 0, 0, 0],
    #         [0, 1, 2, 0],
    #         [0, 3, 4, 0],
    #         [0, 0, 0, 0]])
```

In [ ]:
def zero_pad_2d(image):
    '''2차원 image 둘레에 값 0인 행과 열을 한 줄씩 추가해 반환합니다.'''
    # TODO: 원본보다 높이와 너비가 각각 2 큰 배열을 만들고 가운데에 image를 넣으세요.
    return None

### 채점머신 - Q8
함수를 구현한 다음 아래 셀을 실행해주세요. 채점 코드는 수정하지 않습니다.

In [ ]:
def check_zero_pad_2d():
    image = np.array([[1, 2], [3, 4]])
    expected = np.array([
        [0, 0, 0, 0],
        [0, 1, 2, 0],
        [0, 3, 4, 0],
        [0, 0, 0, 0],
    ])
    actual = zero_pad_2d(image)
    assert isinstance(actual, np.ndarray), "NumPy 배열을 반환해야 합니다."
    assert actual.shape == (4, 4), f"출력 shape은 (4, 4)여야 하나 {actual.shape}입니다."
    assert np.array_equal(actual, expected), f"예상:\n{expected}\n실제:\n{actual}"
    assert np.array_equal(image, [[1, 2], [3, 4]]), "입력 image를 변경하지 마세요."
    assert zero_pad_2d(np.array([[7]])).shape == (3, 3), "1x1 입력도 확인해보세요."

try:
    check_zero_pad_2d()
    print("🎉 zero_pad_2d 정답입니다!")
except (AssertionError, TypeError, AttributeError) as error:
    print("다시 확인해주세요:", error)

## Q9. 3x3 convolution 구현하기

Q7에서 heatmap으로 봤던 kernel을 이번에는 실제로 움직여봅시다.  
`conv3x3_valid(image, kernel)`은 왼쪽 위부터 3x3 patch를 한 칸씩 옮기며 계산합니다.  
각 위치에서 `patch * kernel`의 9개 값을 더해 출력 한 칸에 기록해주세요.  
6x6 입력이라면 출력은 4x4가 됩니다.

- **입력:** 2차원 배열 `image`와 3x3 배열 `kernel`.
- **출력:** 각 3x3 patch의 곱셈 합을 담은 `(높이-2, 너비-2)` 배열. 원본 이미지를 바꾸지 않습니다.

아래 예시에서는 모든 kernel 값이 1이므로, 각 출력 칸은 해당 3x3 영역의 합입니다.

```python
    image = np.array([[ 1,  2,  3,  4],
                    [ 5,  6,  7,  8],
                    [ 9, 10, 11, 12],
                    [13, 14, 15, 16]])
    kernel = np.ones((3, 3))
    conv3x3_valid(image, kernel)
    #  = array([[54., 63.],
    #          [90., 99.]])
```

코드에서 `np.sum(patch * kernel)`을 사용해도 됩니다. 필터를 뒤집거나 ReLU를 적용하지 않습니다.

In [ ]:
def conv3x3_valid(image, kernel):
    '''2차원 image에 3x3 kernel을 stride 1, padding 없이 적용합니다.'''
    # TODO: 출력 배열을 만들고 각 위치의 3x3 patch와 kernel을 곱해 더하세요.
    return None

### 채점머신 - Q9
정답 값뿐 아니라 출력 shape과 kernel 방향도 확인합니다.

In [ ]:
def check_conv3x3_valid():
    expected = np.array([
        [3, 3, -3, -3],
        [3, 3, -3, -3],
        [3, 3, -3, -3],
        [3, 3, -3, -3],
    ], dtype=np.float32)
    actual = conv3x3_valid(small_image, kernel)
    assert isinstance(actual, np.ndarray), "NumPy 배열을 반환해야 합니다."
    assert actual.shape == (4, 4), f"출력 shape은 (4, 4)여야 하나 {actual.shape}입니다."
    assert np.allclose(actual, expected), f"예상:\n{expected}\n실제:\n{actual}"

    image = np.arange(1, 10, dtype=np.float32).reshape(3, 3)
    asymmetric_kernel = np.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]])
    assert np.allclose(conv3x3_valid(image, asymmetric_kernel), [[-8]]), \
        "kernel을 뒤집지 않고 같은 위치끼리 곱하는지 확인해보세요."

try:
    check_conv3x3_valid()
    print("🎉 conv3x3_valid 정답입니다!")
except (AssertionError, TypeError, AttributeError) as error:
    print("다시 확인해주세요:", error)

### Q8과 Q9를 연결해봅시다

Q8에서는 이미지 둘레에 0을 한 줄 더했고, Q9에서는 3x3 kernel을 padding 없이 적용했습니다.  
2개의 함수를 같이 사용하여, padding 옵션이 있는 Conv 를 구현할 수 있습니다.  
**아래에 제공된 코드를 바탕으로 살펴봅시다**  

이번에는 `padding=0`과 `padding=1`만 다룹니다. `padding=1`이면 Q8의 함수를 먼저 쓰고, 그 결과에 Q9의 함수를 적용합니다.

- **입력:** 2차원 `image`, 3x3 `kernel`, `padding` 값 0 또는 1.
- **출력:** convolution 결과 배열. 6x6 입력이라면 padding 0에서 4x4, padding 1에서 6x6입니다.

Q8, Q9의 채점머신을 통과했다면 아래 셀을 실행해주세요.


In [ ]:
def conv3x3_with_padding(image, kernel, padding=0):
    """Q8의 zero_pad_2d와 Q9의 conv3x3_valid를 연결한 제공 함수입니다."""
    if padding == 0:
        prepared_image = image
    elif padding == 1:
        prepared_image = zero_pad_2d(image)
    else:
        raise ValueError("이번에는 padding 0 또는 1만 사용합니다.")

    return conv3x3_valid(prepared_image, kernel)


이제 Q7의 `small_image`에 실제로 적용해봅시다.  
아래 두 결과는 **같은 이미지, 같은 kernel**을 사용합니다. padding만 바꿨을 때 배열의 크기와 가장자리 값이 어떻게 달라지는지 살펴보세요.


In [ ]:
valid_map = conv3x3_with_padding(small_image, kernel, padding=0)
padded_map = conv3x3_with_padding(small_image, kernel, padding=1)

fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout="constrained")
color_limit = max(1.0, float(np.max(np.abs(valid_map))), float(np.max(np.abs(padded_map))))
show_values(axes[0], valid_map, "padding 0", cmap="coolwarm", vmin=-color_limit, vmax=color_limit)
show_values(axes[1], padded_map, "padding 1", cmap="coolwarm", vmin=-color_limit, vmax=color_limit)
plt.show()

print("padding 0:", valid_map.shape)
print("padding 1:", padded_map.shape)


그림에서 한 칸씩 계산되는 흐름도 살펴봅시다.  
아래 코드는 `matplotlib`으로 GIF를 만드는 **제공 코드**입니다. 애니메이션 코드 자체를 이해하거나 수정할 필요는 없습니다.  
주황색 테두리가 현재 보는 3x3 patch이고, 오른쪽 feature map은 계산된 칸부터 채워집니다. 회색 칸은 아직 계산하지 않은 위치입니다.  
GIF를 만드는 데 잠시 시간이 걸릴 수 있습니다.


In [ ]:
# 그래프를 GIF로 만드는 제공 코드입니다. 자세하게 보지 않아도 됩니다!
from pathlib import Path
from tempfile import TemporaryDirectory
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Rectangle
from IPython.display import Image, display

cases = [
    (0, small_image, conv3x3_with_padding(small_image, kernel, padding=0)),
    (1, zero_pad_2d(small_image), conv3x3_with_padding(small_image, kernel, padding=1)),
]
frames = []
for case_index, (_, _, feature_map) in enumerate(cases):
    frames.extend((case_index, step) for step in range(feature_map.size))
    frames.extend([(case_index, feature_map.size - 1)] * 3)  # 마지막 결과를 잠깐 보여줍니다.

gif_cmap = plt.get_cmap("coolwarm").copy()
gif_cmap.set_bad("#eeeeee")  # 아직 계산하지 않은 출력 칸
gif_limit = max(1.0, max(float(np.max(np.abs(result))) for _, _, result in cases))
fig, axes = plt.subplots(1, 3, figsize=(11, 4), layout="constrained")


def draw_gif_grid(ax, values, title, *, cmap, vmin, vmax):
    ax.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(f"{title} ({values.shape[0]}x{values.shape[1]})")
    ax.set_xticks(range(values.shape[1]))
    ax.set_yticks(range(values.shape[0]))
    ax.set_xticks(np.arange(-0.5, values.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, values.shape[0], 1), minor=True)
    ax.grid(which="minor", color="lightgray", linewidth=0.8)
    ax.tick_params(which="both", length=0, labelsize=8)
    for row in range(values.shape[0]):
        for col in range(values.shape[1]):
            value = values[row, col]
            if np.isnan(value):
                continue
            color = "white" if (value < 0.5 if cmap == "gray" else abs(value) > 0.65 * max(abs(vmin), abs(vmax))) else "black"
            ax.text(col, row, f"{value:g}", ha="center", va="center", color=color, fontsize=8)


def draw_gif_frame(frame):
    case_index, step = frame
    padding, prepared_image, feature_map = cases[case_index]
    row, col = divmod(step, feature_map.shape[1])
    shown = np.full(feature_map.shape, np.nan)
    shown.flat[:step + 1] = feature_map.flat[:step + 1]

    for ax in axes:
        ax.clear()
    draw_gif_grid(axes[0], prepared_image, f"input, padding {padding}", cmap="gray", vmin=0, vmax=1)
    draw_gif_grid(axes[1], kernel, "kernel", cmap=gif_cmap, vmin=-1, vmax=1)
    draw_gif_grid(axes[2], shown, "feature map", cmap=gif_cmap, vmin=-gif_limit, vmax=gif_limit)
    axes[0].add_patch(Rectangle((col - 0.5, row - 0.5), 3, 3, fill=False, edgecolor="#E69F00", linewidth=3))
    axes[2].add_patch(Rectangle((col - 0.5, row - 0.5), 1, 1, fill=False, edgecolor="#00A6A6", linewidth=3))
    fig.suptitle(f"padding={padding}, output[{row}, {col}] = sum(patch * kernel) = {feature_map[row, col]:g}")


animation = FuncAnimation(fig, draw_gif_frame, frames=frames, interval=250, repeat=True)
with TemporaryDirectory() as temp_dir:
    gif_path = Path(temp_dir) / "convolution_padding.gif"
    animation.save(str(gif_path), writer=PillowWriter(fps=4), dpi=90)
    display(Image(data=gif_path.read_bytes()))
plt.close(fig)


## Q10. 2x2 max pooling 구현하기

이번에는 feature map을 2x2씩 묶어봅시다.  
`max_pool2x2(feature_map)`은 겹치지 않는 영역에서 **가장 큰 값 하나**만 남깁니다. 입력의 높이와 너비는 모두 짝수라고 가정합니다.

- **입력:** 높이와 너비가 모두 짝수인 2차원 배열 `feature_map`.
- **출력:** 겹치지 않는 각 2x2 영역의 최댓값을 담은 `(높이/2, 너비/2)` 배열.

아래 예시의 아래쪽 두 영역은 모두 음수입니다. 이때도 실제 최댓값을 남겨야 합니다.

```python
    feature_map = np.array([[ 1,  5,  0,  2],
                            [ 3,  4,  7,  6],
                            [-1, -3, -8, -2],
                            [-5, -4, -6, -7]])
    max_pool2x2(feature_map)
    # = array([[ 5,  7],
    #         [-1, -2]])
```

음수만 있는 영역도 있으니 출력 배열을 0으로 초기화한 뒤 `max`로 갱신하는 방식은 주의하세요.

In [ ]:
def max_pool2x2(feature_map):
    '''짝수 크기의 2차원 feature_map에 2x2 max pooling, stride 2를 적용합니다.'''
    # TODO: 2x2 영역의 최댓값을 출력 배열의 한 칸에 기록하세요.
    return None

### 채점머신 - Q10

In [ ]:
def check_max_pool2x2():
    feature_map = np.array([
        [1, 4, 2, 0],
        [3, 2, 5, 1],
        [0, 1, 3, 2],
        [2, 0, 1, 6],
    ])
    expected = np.array([[4, 5], [2, 6]])
    actual = max_pool2x2(feature_map)
    assert isinstance(actual, np.ndarray), "NumPy 배열을 반환해야 합니다."
    assert actual.shape == (2, 2), f"출력 shape은 (2, 2)여야 하나 {actual.shape}입니다."
    assert np.array_equal(actual, expected), f"예상:\n{expected}\n실제:\n{actual}"
    negatives = np.array([[-5, -2], [-3, -4]])
    assert np.array_equal(max_pool2x2(negatives), [[-2]]), \
        "음수만 있는 영역에서도 실제 최댓값을 남겨야 합니다."

try:
    check_max_pool2x2()
    print("🎉 max_pool2x2 정답입니다!")
except (AssertionError, TypeError, AttributeError) as error:
    print("다시 확인해주세요:", error)

<a id="section-4-models"></a>
# 4. LeNet, AlexNet에서 Fashion-MNIST로

LeNet과 AlexNet이 어떤 층을 어떤 순서로 쌓았는지 먼저 살펴봅시다.  
두 모델의 층 구성을 비교할 때 아래 이미지를 참고해주세요. 이후에는 28x28 흑백 이미지에 맞게 크기를 줄인 모델을 만들어봅니다.  
아래 실습의 LeNet-like와 AlexNet-like는 원본 구조를 그대로 재현한 모델은 아닙니다.

### Q11. LeNet과 AlexNet 구조 살펴보기

아래 그림에서 convolution, pooling, Dense가 어떤 순서로 등장하는지 살펴봅시다. 그림은 구조 비교를 위한 요약이며, 아래에서 구현할 축소 모델의 세부 수치와는 다릅니다.  
그림의 LeNet 입력은 28x28로 표시되어 있지만 [원 논문의 LeNet-5](https://leon.bottou.org/papers/lecun-98h)는 32x32 입력을 사용합니다.

<img src="./imgs/lenet_alexnet.png" width=600>

출처: https://en.wikipedia.org/wiki/AlexNet#History

## 데이터 준비와 비교 조건

아래 코드는 1주차에서 사용한 Fashion-MNIST를 다시 불러옵니다.  

`load_data()`는 학습용, 평가용 이미지와 정답을 반환합니다.  
`astype("float32") / 255.0`은 픽셀 값을 0~1 사이로 바꾸고, `[..., np.newaxis]`는 맨 뒤에 흑백 채널 축을 추가합니다.  
따라서 이미지 한 장의 shape는 `(28, 28)`에서 `(28, 28, 1)`로 바뀝니다.  
코드를 실행해보고, shape 가 맞는지 확인해보세요.

Dense, 단순 CNN, LeNet-like, AlexNet-like 네 모델은 **같은 데이터, optimizer, batch size, epoch, validation 비율**로 학습합니다.  
Test set은 마지막 비교에 사용하고, 모델을 수정할 때는 validation 결과를 먼저 봅니다.

In [ ]:
(train_input, train_target), (test_input, test_target) \
    = keras.datasets.fashion_mnist.load_data()
train_images = (train_input.astype("float32") / 255.0)[..., np.newaxis]
test_images = (test_input.astype("float32") / 255.0)[..., np.newaxis]

print("학습 입력:", train_images.shape, "정답:", train_target.shape)
print("평가 입력:", test_images.shape, "정답:", test_target.shape)

## Q12. Keras Layer로 모델 구성하기

아래 함수에서는 2주차에 사용했던 방식대로 Keras Layer를 순서대로 쌓습니다.  
Keras 의 다음의 함수를 통해 구현할 수 있습니다.

- `keras.Sequential([...])`
- `keras.Input(shape=(28, 28, 1))`
- `keras.layers.Conv2D(filters, kernel_size, padding=..., activation=...)`
- `keras.layers.MaxPooling2D(2)`, `keras.layers.AveragePooling2D(2)`
- `keras.layers.Flatten()`
- `keras.layers.Dense(units, activation=...)`
- `keras.layers.Dropout(0.3)`

아래의 `build_dense()`와 `build_simple_cnn()`은 Keras 코드 예시로 제공됩니다.  
두 함수를 참고해서 `build_lenet_like()`와 `build_alexnet_like()`를 작성해주세요.  
사용할 Layer와 순서는 **함수 안의 주석**에 적었습니다. 원본 모델을 그대로 옮기는 대신 28x28 흑백 이미지에 맞게 조정했습니다.

각 함수는 인자 없이 호출하면 Keras 모델을 반환합니다. 입력은 이미지 한 장당 `(28, 28, 1)`이고, 마지막에는 옷 종류 10개의 확률을 출력합니다.  
먼저 코드를 작성하고, 그다음 아래에서 지정한 층의 출력 shape를 예상해봅시다.

In [ ]:
def build_dense():
    return keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        keras.layers.Flatten(),
        keras.layers.Dense(10, activation="softmax"),
    ], name="dense")


def build_simple_cnn():
    return keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Flatten(),
        keras.layers.Dense(10, activation="softmax"),
    ], name="simple_cnn")


def build_lenet_like():
    # 입력: 28x28 흑백 이미지
    # 1. 5x5 Conv2D 필터 6개, same, ReLU -> 2x2 AveragePooling2D
    # 2. 5x5 Conv2D 필터 16개, valid, ReLU -> 2x2 AveragePooling2D
    # 3. Flatten -> Dense(120, ReLU) -> Dense(84, ReLU) -> Dense(10, softmax)
    # TODO: 위 순서대로 keras.Sequential 모델을 만들어 반환하세요.
    return None


def build_alexnet_like():
    # 입력: 28x28 흑백 이미지
    # 1. 3x3 Conv2D 필터 16개, same, ReLU -> 2x2 MaxPooling2D
    # 2. 3x3 Conv2D 필터 32개, same, ReLU -> 2x2 MaxPooling2D
    # 3. 3x3 Conv2D 필터 32개, same, ReLU -> 같은 설정의 Conv2D 한 번 더
    # 4. 3x3 Conv2D 필터 64개, same, ReLU
    # 5. Flatten -> Dense(64, ReLU) -> Dropout(0.3) -> Dense(10, softmax)
    # TODO: 위 순서대로 keras.Sequential 모델을 만들어 반환하세요.
    return None


### 코드 작성 후 출력 shape 예상하기

완성한 코드를 보고 아래 다섯 위치의 출력 shape를 예상해주세요.  
**이미지 한 장**을 입력했을 때를 기준으로, 높이, 너비, 채널 수 또는 벡터 길이를 적어주세요.

| 모델 | Layer | 예상 출력 shape |
|---|---|---|
| 단순 CNN | `MaxPooling2D(2)` 직후 | |
| LeNet-like | 두 번째 `Conv2D(16, 5, padding="valid")` 직후 | |
| LeNet-like | 마지막 `Dense(10, softmax)` 직후 | |
| AlexNet-like | `Flatten()` 직후 | |
| AlexNet-like | 마지막 `Dense(10, softmax)` 직후 | |

예상을 적은 다음 아래 채점머신과 `model.summary()`를 실행해 확인해주세요.  
`summary()`에 표시되는 맨 앞 축은 이미지 장수이므로, 이번 표에서는 그 축을 제외하고 비교합니다.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

### 채점머신 - Q12
네 모델의 Layer 종류와 순서, 최종 출력 shape를 확인합니다.

In [ ]:
def check_model_builders():
    dense = build_dense()
    simple_cnn = build_simple_cnn()
    lenet = build_lenet_like()
    alexnet = build_alexnet_like()

    for name, model in [("Dense", dense), ("단순 CNN", simple_cnn),
                        ("LeNet-like", lenet), ("AlexNet-like", alexnet)]:
        assert isinstance(model, keras.Model), f"{name}: Keras 모델을 반환해주세요."
        assert model.output_shape == (None, 10), \
            f"{name}: 최종 출력은 10개 클래스의 확률이어야 합니다. 현재 {model.output_shape}"

    assert [type(layer).__name__ for layer in simple_cnn.layers] == [
        "Conv2D", "MaxPooling2D", "Flatten", "Dense",
    ], "단순 CNN의 Layer 순서를 확인해주세요."
    assert [type(layer).__name__ for layer in lenet.layers] == [
        "Conv2D", "AveragePooling2D", "Conv2D", "AveragePooling2D",
        "Flatten", "Dense", "Dense", "Dense",
    ], "LeNet-like의 Layer 순서와 종류를 확인해주세요."
    assert [type(layer).__name__ for layer in alexnet.layers] == [
        "Conv2D", "MaxPooling2D", "Conv2D", "MaxPooling2D",
        "Conv2D", "Conv2D", "Conv2D", "Flatten", "Dense", "Dropout", "Dense",
    ], "AlexNet-like의 Layer 순서와 종류를 확인해주세요."
    assert simple_cnn.layers[0].output.shape[1:] == (28, 28, 16), \
        "단순 CNN의 첫 Conv2D 출력 shape를 확인해주세요."
    assert lenet.layers[0].output.shape[1:] == (28, 28, 6), \
        "LeNet-like 첫 Conv2D의 filters, kernel_size, padding을 확인해주세요."
    assert alexnet.layers[-2].rate == 0.3, "AlexNet-like dropout rate를 확인해주세요."
    print("🎉 모델 구조를 확인했습니다!")

try:
    check_model_builders()
except (AssertionError, TypeError, AttributeError) as error:
    print("다시 확인해주세요:", error)


문제를 풀고, 아래 코드를 통해 결과를 확인해봅시다.

In [ ]:
models = {
    "Dense": build_dense(),
    "단순 CNN": build_simple_cnn(),
    "LeNet-like": build_lenet_like(),
    "AlexNet-like": build_alexnet_like(),
}
one_image = np.zeros((1, 28, 28, 1), dtype=np.float32)

for name, model in models.items():
    print(f"\n===== {name} =====")
    model.summary()
    print("이미지 한 장의 최종 출력 shape:", tuple(model(one_image)[0].shape))


## Q13. 같은 조건에서 학습하고 정확도 비교하기

`summary()`를 보고 parameter 수가 가장 많은 모델도 적어봅시다. 모델이 크다고 반드시 정확도가 가장 높을까요?  
Dense, 단순 CNN, LeNet-like, AlexNet-like의 validation accuracy가 어떤 순서로 나올지도 예상해주세요.  
필요한 데이터 수는 어느 정도일지, 과소적합이나 과대적합을 각 구조에서 어떻게 줄일 수 있을지도 생각해봅시다.

이 내용을 바탕으로 문제에 맞는 모델을 어떤 기준으로 고를지 도메인, parameter 수, 데이터의 관점에서 정리해주세요.  
작성한 다음 아래 코드로 네 모델을 같은 조건에서 학습해 실제 결과를 확인해봅시다.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

In [ ]:
EPOCHS = 5
BATCH_SIZE = 128
builders = {
    "Dense": build_dense,
    "단순 CNN": build_simple_cnn,
    "LeNet-like": build_lenet_like,
    "AlexNet-like": build_alexnet_like,
}
trained_models = {}
histories = {}

for name, builder in builders.items():
    tf.keras.utils.set_random_seed(42)
    model = builder()
    model.compile(
        optimizer=keras.optimizers.RMSprop(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    print(f"\n===== {name} =====")
    history = model.fit(
        train_images, train_target,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        verbose=2,
    )
    trained_models[name] = model
    histories[name] = history.history

print("\nValidation 결과")
for name, history in histories.items():
    print(name, "마지막 val_accuracy:", round(history["val_accuracy"][-1], 4),
          "parameter:", trained_models[name].count_params())


네 모델의 validation 결과를 먼저 비교해주세요. 모델별 설정을 바꿔보고 싶다면 이 결과를 근거로 실험한 뒤, 아래 test 셀은 최종 비교에 사용합니다.  
**미니문제(필수 아님):** 같은 학습 설정을 쓴 네 모델 중 하나의 설정을 바꾸고 validation 결과가 어떻게 달라지는지 살펴보세요.


In [ ]:
best_cnn_name = max(
    ("단순 CNN", "LeNet-like", "AlexNet-like"),
    key=lambda name: histories[name]["val_accuracy"][-1],
)
print("Validation 기준 선택된 CNN:", best_cnn_name)

test_results = {}
for name, model in trained_models.items():
    test_loss, test_accuracy = model.evaluate(test_images, test_target, verbose=0)
    test_results[name] = test_accuracy
    print(f"{name}: test accuracy = {test_accuracy:.4f} / loss = {test_loss:.4f}")

print("Dense 대비 차이:", round(test_results[best_cnn_name] - test_results["Dense"], 4))


**결과 정리**

| 모델 | parameter 수 | validation accuracy | test accuracy |
|---|---:|---:|---:|
| Dense | | | |
| 단순 CNN | | | |
| LeNet-like | | | |
| AlexNet-like | | | |

예상과 달랐던 결과가 있다면 어떤 점이 달랐는지 적어주세요. 네 모델 중 이번 데이터에서 어떤 모델을 고를지도 근거와 함께 정리해주세요.


**>>> 이 멘트를 지우고 답변을 작성해주세요. <<<**

# 정리

이번 주차에서는 본격적으로 특정 도메인을 다뤄보았습니다.  
그 첫번째로 이미지를 다루게 되었는데, 이미지 자체가 어떤 특성을 가지고 있는지 이해하고, 기본적인 CNN 관련 구조를 살펴보았습니다.

다음주에는 좀 더 발전된 아키텍처와 이미지 도메인에 맞는 데이터, 모델 처리방법을 알아보겠습니다.  
4주차 때 뵙겠습니다!

<a id="advanced-channels"></a>
# 선택 심화 A: 여러 채널과 kernel bank

필수 과제에서는 흑백 이미지의 채널 하나에 kernel 하나를 적용했습니다.  
하지만 일반적인 RGB 이미지는 같은 위치에 R, G, B 값이 있습니다.  
이를 처리하는 방식에는 `Channel` 이라는 개념이 사용됩니다.  

`Channel` 은 같은 위치에서 서로 다른 정보를 표현하게 됩니다. RGB 이미지에서는 R, G, B 각각의 값을 의미하도록 설정해줄 수 있습니다.
CNN의 중간 feature map도 채널을 여러 개 가질 수 있습니다.  

입력 채널이 $C_{in}$개일 때 **출력 필터 하나** 는 각 입력 채널에 대응하는 작은 kernel을 하나씩 가집니다.  
같은 위치에서 채널별 계산 결과를 모두 더해 출력 값 하나를 만듭니다.  
이런 출력 필터를 $C_{out}$개 모은 것이 **kernel bank**이며, 출력 feature map도 $C_{out}$개가 됩니다.

Keras의 `Conv2D` 에서 3x3 kernel bank의 가중치 shape는 `(3, 3, C_in, C_out)`입니다.
아래 코드는 입력 채널 두 개와 출력 필터 세 개를 사용하는 예시입니다. 입력, kernel bank, 출력의 shape 등을 집중해서 살펴보세요.

**더 읽어보기:** [MIT Vision Book, 24.2.1 Multi-Input, Multi-Output Convolutional Layers](https://visionbook.mit.edu/convolutional_neural_nets.html)의 **Kernel**과 **Kernel Bank** 설명을 읽어보길 추천합니다.  
읽으면서 "입력 채널 수와 출력 feature map 수를 각각 무엇이 결정하는가?"를 스스로 확인해보세요.
이 책은 채널 축을 앞에 두는 표기도 사용하므로, 위의 Keras shape와 축 순서가 달라도 같은 개념으로 읽으면 됩니다.

In [ ]:
# 입력 채널 2개, 출력 필터 3개인 Conv2D 예시
channel_image = np.zeros((4, 4, 2), dtype=np.float32)
channel_image[:, :, 0] = 1.0
channel_image[0, 0, 1] = 2.0

bank_layer = keras.layers.Conv2D(filters=3, kernel_size=3, padding="valid")
# NumPy 배열을 TensorFlow Tensor로 바꿔 Conv2D에 전달합니다.
channel_tensor = tf.convert_to_tensor(channel_image[np.newaxis, ...])
feature_maps = bank_layer(channel_tensor)[0]

print("입력 한 장:", channel_image.shape)
print("kernel bank:", tuple(bank_layer.kernel.shape))
print("bias:", tuple(bank_layer.bias.shape))
print("출력 한 장:", tuple(feature_maps.shape))


<a id="advanced"></a>
# 선택 심화 B: NumPy 계산과 Keras Custom Layer

다음 두 코드는 모두 여러 필터로 feature map을 만듭니다.  
첫 번째는 필수 과제에서 작성한 `conv3x3_valid`를 바탕으로 여러 Kernel 을 적용합니다. Kernel 별로 결과를 저장합니다.  
두 번째는 해당 기능을 Keras의 `Layer`로 옮기고, `add_weight()`로 학습할 kernel 과 bias 를 등록합니다.  

In [ ]:
# 필수 과제의 conv3x3_valid를 필터마다 한 번씩 사용합니다.
def conv_layer_2d(image, kernels, biases):
    feature_maps = [
        conv3x3_valid(image, kernel) + bias
        for kernel, bias in zip(kernels, biases)
    ]
    return np.stack(feature_maps, axis=-1)


example_image = np.ones((3, 3), dtype=np.float32)
example_kernels = np.stack([np.ones((3, 3)), np.zeros((3, 3))])
example_biases = np.array([0, 2])
print("NumPy 결과:", conv_layer_2d(example_image, example_kernels, example_biases))


Keras의 Custom Layer에서는 `build()`가 입력 채널 수를 알고 가중치를 만드는 역할을 하고, `call()`이 실제 출력을 계산합니다.  
아래 예시는 `tf.image.extract_patches`로 각 3x3 영역을 꺼낸 다음 kernel과 곱해 더합니다.  
TensorFlow 연산을 유지해야 loss에서 가중치까지 gradient가 이어집니다.  

참고: [Keras, Making new layers and models via subclassing](https://keras.io/guides/making_new_layers_and_models_via_subclassing/)


In [ ]:
class CustomConv2D(keras.layers.Layer):
    def __init__(self, filters, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters

    def build(self, input_shape):
        in_channels = int(input_shape[-1])
        self.kernel = self.add_weight(
            name="kernel",
            shape=(3, 3, in_channels, self.filters),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.bias = self.add_weight(
            name="bias", shape=(self.filters,), initializer="zeros", trainable=True,
        )

    def call(self, inputs):
        patches = tf.image.extract_patches(
            inputs, sizes=[1, 3, 3, 1], strides=[1, 1, 1, 1],
            rates=[1, 1, 1, 1], padding="VALID",
        )
        flat_kernel = tf.reshape(self.kernel, (-1, self.filters))
        return tf.einsum("bhwc,cf->bhwf", patches, flat_kernel) + self.bias


custom_layer = CustomConv2D(filters=2)
one_image = tf.ones((1, 5, 5, 1))
custom_output = custom_layer(one_image)
print("출력 한 장:", tuple(custom_output[0].shape))
print("학습할 가중치:", [(w.name, tuple(w.shape)) for w in custom_layer.trainable_weights])


## trainable 가중치는 어떻게 업데이트될까요?

2주차에는 Keras가 가중치를 추적한다는 점을 살펴봤습니다. 이번에는 **한 번의 학습 단계에서 값이 어떻게 바뀌는지**를 따라가봅시다.  
`CustomConv2D`의 `add_weight()`로 등록한 변수는 `trainable_weights`에 들어갑니다.  
`GradientTape`가 출력과 loss 계산을 기록하고, loss에 대한 각 가중치의 gradient를 구합니다.  
optimizer가 그 gradient를 사용해 변수 값을 업데이트합니다. 아래 코드는 이 과정을 한 번 실행합니다.


In [ ]:
optimizer = keras.optimizers.SGD(learning_rate=0.01)
before = [weight.numpy().copy() for weight in custom_layer.trainable_weights]

with tf.GradientTape() as tape:
    prediction = custom_layer(one_image)
    loss = tf.reduce_mean(prediction ** 2)

weights = custom_layer.trainable_weights
gradients = tape.gradient(loss, weights)
optimizer.apply_gradients(zip(gradients, weights))

for weight, old_value in zip(weights, before):
    print(weight.name, "업데이트 크기:", np.max(np.abs(weight.numpy() - old_value)))


**선택 정리:** 위 코드에서 `trainable_weights → gradient → optimizer`가 이어지는 과정을 살펴보세요.  
`conv_layer_2d`의 NumPy 배열은 Keras의 학습 변수로 등록되지 않지만, `CustomConv2D`의 `kernel`과 `bias`는 `add_weight()`로 등록됩니다.  
한 번의 학습 단계에서 TensorFlow가 어떤 연산을 기록하고, gradient를 어떻게 구하며, SGD가 그 값을 사용해 가중치를 어떻게 바꾸는지 따라가보세요.

참고 자료: [TensorFlow 자동 미분과 GradientTape](https://www.tensorflow.org/guide/autodiff), [TensorFlow 변수](https://www.tensorflow.org/guide/variable), [TensorFlow Core API로 optimizer 구현](https://www.tensorflow.org/guide/core/optimizers_core)
